# Challenge Exam · Team 04

MA2015.202 · Planta Física Challenge · Written team code examination

**Team:** `MA2015-202-04` (Bolaños Valencia, Duan, García Fuentes)

Friday, October 23, 2026

## Instructions

1. Answer the four questions. Each question has a Markdown cell **Answer** below it: write your answer there, in text.
2. You may add code cells to compute or test code that supports your answer. If you use them, run them so the output is visible.
3. Use your final submission (`planta_fisica_minimum_solution.ipynb`): cells are numbered starting at 0 in the order they appear in the file (the first cell is cell 0). A PDF copy, `Final-MA2015.202-equipo-04.pdf`, is included in this zip.
4. Questions 1 to 3 come from your challenge evaluation (the number in parentheses); question 4 was written by the Sustainable Development Engineering (IDS) colleagues.
5. **Submission:** export this notebook to **PDF** and upload it to the Canvas assignment “Written Team Code Assessment”. One submission per team. The due time is today at 18:00; late submissions are accepted until 23:59 today.

## Question 1 · Research (1.5)

Look at `evaluate(genes)` in cell 8 (lines 5--7). What are the two distinct validation checks performed before `treated` is computed, and how does the treatment of a domain violation ($k_i < 0$ or $k_i > u_i$) differ from the treatment of a plant capacity violation ($T(k) > 1410\text{ m}^3$)?

### Answer

Before computing `treated`, `evaluate` makes two checks:

1. **Shape and type:** the chromosome must have exactly 4 integer genes. If not, it raises a `ValueError`.
2. **Domain:** each gene must be between 0 and its limit, `LIMITS = (8, 8, 6, 8)`. If not, it raises a `ValueError`.

**Difference:**

- A **domain violation** is treated as an invalid input. The function stops with an error and no fitness is computed, because something like $k_3 = 7$ (70 % in Cedetec) or a negative level is not a valid plan. In practice this never happens during the GA, since `random_individual` and `mutate` always stay inside the domains.
- A **capacity violation** ($T > 1410$ m³) is a valid plan that is simply worse. It is fully evaluated, marked `feasible = False`, and gets a penalty fitness $F = 1 + E/1410$, where $E = T - 1410$. Feasible plans have $F \le 1$, so an infeasible plan always ranks behind them, but it stays in the population. For example, $(6,6,4,8)$ has $T = 1445.484$ m³ and $F = 1.02517$.

## Question 2 · Development (2.3)

In cell 28 you trace one generation on the 4-individual demonstration population from `demo_rng = np.random.default_rng(7)`, and in cell 29 you highlight `slot 2`: parents $[4, 7, 5, 7]$ and $[3, 7, 0, 6]$ cross with mask `BBAA` to produce $[3, 7, 5, 7]$, which then mutates to $[4, 7, 5, 7]$ with $F = 0.39104$. Walk through what happens in `mutate` (cell 21) during that transition $[3, 7, 5, 7] \to [4, 7, 5, 7]$, and explain why the final mutated child ends up identical to Parent A even though the crossover mask was `BBAA`.

In [1]:
# Replay slot 2 of the cell-29 trace and expose every random draw made inside mutate (cell 21).
import contextlib, io, json
import numpy as np
import matplotlib
matplotlib.use("Agg")

final_nb = json.load(open("planta_fisica_minimum_solution.ipynb"))
ns = {}
with contextlib.redirect_stdout(io.StringIO()):
    for cell in final_nb["cells"][:28]:  # problem data and operators: mutate (cell 21) and uniform_crossover (cell 27)
        if cell["cell_type"] == "code":
            exec("".join(cell["source"]), ns)
evaluate, LIMITS = ns["evaluate"], ns["LIMITS"]

rng = np.random.default_rng(42)                       # the generator used in cell 29
population = [ns["random_individual"](rng) for _ in range(4)]
for slot in (1, 2):
    a, b = ns["tournament"](population, rng), ns["tournament"](population, rng)
    child = ns["uniform_crossover"](a, b, rng)
    if slot == 1:
        ns["mutate"](child, 0.25, rng)                # consume slot 1's draws exactly as next_generation does
print("Parent A", a, "| Parent B", b, "| child", child, f"F={evaluate(child)['fitness']:.5f}")

result = child.copy()                                 # same loop as mutate, with each draw printed
for i, limit in enumerate(LIMITS):
    u = rng.random()
    if u < 0.25:
        neighbors = [int(v) for v in (result[i] - 1, result[i] + 1) if 0 <= v <= limit]
        result[i] = rng.choice(neighbors)
        print(f"k{i + 1}: u={u:.4f} < 0.25 -> selected, neighbors {neighbors} -> {result[i]}")
    else:
        print(f"k{i + 1}: u={u:.4f} >= 0.25 -> kept at {result[i]}")
m = evaluate(result)
print("mutant", result, f"T={m['treated']:.3f} m3  F={m['fitness']:.5f}  identical to Parent A: {np.array_equal(result, a)}")
print("Plans in the whole space with F = 0.39104:",
      [k for k in np.ndindex(*(LIMITS + 1)) if abs(evaluate(np.array(k))["fitness"] - 0.39104) < 5e-6])

Parent A [6 6 5 7] | Parent B [3 7 0 6] | child [3 7 5 7] F=0.47382
k1: u=0.0438 < 0.25 -> selected, neighbors [2, 4] -> 4
k2: u=0.6830 >= 0.25 -> kept at 7
k3: u=0.7448 >= 0.25 -> kept at 5
k4: u=0.9675 >= 0.25 -> kept at 7
mutant [4 7 5 7] T=1282.405 m3  F=0.43661  identical to Parent A: False
Plans in the whole space with F = 0.39104: []


### Answer

**Note:** the question does not match our notebook. Our trace (cell 29) uses `default_rng(42)`, and in slot 2 Parent A is $[6,6,5,7]$, not $[4,7,5,7]$. The final mutant $[4,7,5,7]$ has $F = 0.43661$, so it is **not** identical to Parent A, and no plan has $F = 0.39104$. The code cell above reproduces this.

**What `mutate` does to $[3,7,5,7]$ with $p_m = 0.25$:**

- It copies the child and goes through the 4 genes. For each gene it draws a random number, and the gene mutates only if the number is below 0.25.
- $k_1$: draw 0.0438 < 0.25, so it mutates. Its neighbors are 2 and 4, and 4 is chosen.
- $k_2$, $k_3$, $k_4$: draws 0.683, 0.745 and 0.968, so they stay the same.
- Result: $[4,7,5,7]$. Aulas 1 goes from 30 % to 40 %, which adds 84.7 m³ of treated water ($T$: 1197.705 → 1282.405 m³), and $F$ improves from 0.47382 to 0.43661.

**Why a child could end up equal to Parent A:** the mask only says which parent each gene came from. If Parent A really were $[4,7,5,7]$, the `BBAA` child $[3,7,5,7]$ would differ from A only in $k_1$ (3 vs 4). One +1 mutation on $k_1$ would then give back exactly A. (In that case $k_2$ would also be equal in both parents, so our trace would print the mask as `B=AA`.)

## Question 3 · Final (3.3)

In cells 45--49 (Section 16, C2) you compare $p_m = 0.25$ and $p_m = 0.50$ on the uniform-crossover GA (`seed=42`). Both runs reach the exact same final chromosome $(6, 4, 6, 8)$ ($F = 0.38058$), but $p_m = 0.25$ reaches it at generation 8 while $p_m = 0.50$ reaches an intermediate step $(6, 7, 1, 8)$ ($F = 0.38212$) at generation 15 and $(6, 4, 6, 8)$ at generation 16. In cell 47 you explain this delay using the binomial probability of a single-gene improving mutation on a promising 4-gene parent such as $(6, 4, 5, 8)$. Derive those two probabilities for $p_m = 0.25$ and $p_m = 0.50$ and explain why doubling $p_m$ actually *lowers* the chance of that clean single-gene step.

In [2]:
# Exact mutation outcomes of the promising child (6,4,5,8): every gene is kept, moved -1 or moved +1.
from itertools import product

DEMAND = np.array([847.0, 462.0, 284.21, 683.0]); CAP = 1410.0
F = lambda k: (DEMAND.sum() - DEMAND @ k / 10) / DEMAND.sum() if DEMAND @ k / 10 <= CAP + 1e-8 else 1 + (DEMAND @ k / 10 - CAP) / CAP
parent = np.array([6, 4, 5, 8])
print(f"parent {tuple(int(v) for v in parent)} F={F(parent):.5f}")
for i in range(4):  # every single-gene step
    for d in (-1, 1):
        k = parent.copy(); k[i] += d
        if 0 <= k[i] <= LIMITS[i]:
            print(f"  k{i + 1} {d:+d} -> {tuple(int(v) for v in k)}  T={DEMAND @ k / 10:8.3f}  F={F(k):.5f}")


def outcome_probabilities(pm):
    probs = {}
    for moves in product((0, -1, 1), repeat=4):  # 0 = gene not selected
        p, k = 1.0, parent.copy()
        for i, d in enumerate(moves):
            nbrs = [v for v in (parent[i] - 1, parent[i] + 1) if 0 <= v <= LIMITS[i]]
            if d == 0:
                p *= 1 - pm
            elif parent[i] + d in nbrs:
                p *= pm / len(nbrs); k[i] += d
            else:
                p = 0.0
        probs[tuple(int(v) for v in k)] = probs.get(tuple(int(v) for v in k), 0) + p
    return probs


for pm in (0.25, 0.50):
    pr = outcome_probabilities(pm)
    print(f"pm={pm:.2f}: formula pm*1/2*(1-pm)^3 = {pm * 0.5 * (1 - pm) ** 3:.5f} | exact P(->(6,4,6,8)) = {pr[(6, 4, 6, 8)]:.5f}"
          f" | P(any F < 0.38248) = {sum(p for k, p in pr.items() if F(np.array(k)) < 0.38248):.5f}"
          f" | P(child unchanged) = {pr[(6, 4, 5, 8)]:.5f} | P(infeasible) = {sum(p for k, p in pr.items() if F(np.array(k)) > 1):.5f}")

parent (6, 4, 5, 8) F=0.39307
  k1 -1 -> (5, 4, 5, 8)  T=1296.805  F=0.43028
  k1 +1 -> (7, 4, 5, 8)  T=1466.205  F=1.03986
  k2 -1 -> (6, 3, 5, 8)  T=1335.305  F=0.41336
  k2 +1 -> (6, 5, 5, 8)  T=1427.705  F=1.01256
  k3 -1 -> (6, 4, 4, 8)  T=1353.084  F=0.40555
  k3 +1 -> (6, 4, 6, 8)  T=1409.926  F=0.38058
  k4 -1 -> (6, 4, 5, 7)  T=1313.205  F=0.42307
pm=0.25: formula pm*1/2*(1-pm)^3 = 0.05273 | exact P(->(6,4,6,8)) = 0.05273 | P(any F < 0.38248) = 0.05273 | P(child unchanged) = 0.31641 | P(infeasible) = 0.16064
pm=0.50: formula pm*1/2*(1-pm)^3 = 0.03125 | exact P(->(6,4,6,8)) = 0.03125 | P(any F < 0.38248) = 0.03125 | P(child unchanged) = 0.06250 | P(infeasible) = 0.21094


### Answer

From $(6,4,5,8)$, the only single step that improves $F$ is **Cedetec 5 → 6**, which gives $(6,4,6,8)$ with $T = 1409.926 \le 1410$. Raising Aulas 1 or Biblioteca goes over capacity, Aulas 2 is already at its maximum, and any −1 step loses water (see the code output above).

For this to happen, three things are needed:

- Cedetec is chosen to mutate: probability $p_m$.
- It moves up (+1) instead of down: probability $\tfrac12$.
- The other 3 genes do not mutate: probability $(1-p_m)^3$.

$$P = p_m \cdot \tfrac12 \cdot (1-p_m)^3$$

- $p_m = 0.25$: $0.25 \cdot 0.5 \cdot 0.75^3 = 0.0527 \approx 5.3\%$
- $p_m = 0.50$: $0.50 \cdot 0.5 \cdot 0.50^3 = 0.0313 \approx 3.1\%$

**Why doubling $p_m$ lowers it:** doubling $p_m$ makes it twice as likely that Cedetec mutates. But it also makes it much less likely that the other three genes stay still: $0.75^3 = 0.42$ drops to $0.5^3 = 0.125$, more than 3 times smaller. That second effect is bigger, so the clean step becomes less likely. With 4 genes, this probability is highest exactly at $p_m = 1/4$. A higher $p_m$ also changes good plans more often, and it makes them infeasible more often (21 % vs 16 %). That is why $p_m = 0.50$ took until generation 16.

## Question 4 · Interdisciplinary

Your plan shifts 257.4 m³/month more treated water toward Aulas 2 than the historical March 2025 balance, while reducing Biblioteca to 40% reuse. Evaluate whether this change is sustainable when Aulas 2 is considered as a real service destination rather than only an optimization variable. Address hydraulic infrastructure, energy, service equity, and the relevance of the historical 65% reuse reported for Cedetec.

### Answer

| building | March 2025 (observed) | our plan | change |
|---|---|---|---|
| Aulas 1 | 647.0 m³ (76 %) | 508.2 m³ (60 %) | −138.8 |
| Biblioteca | 317.0 m³ (69 %) | 184.8 m³ (40 %) | −132.2 |
| Cedetec | 184.7 m³ (65 %) | 170.5 m³ (60 %) | −14.2 |
| Aulas 2 | 289.0 m³ (42 %) | 546.4 m³ (80 %) | **+257.4** |
| total | 1437.7 m³ | 1409.9 m³ | −27.8 |

**Main point:** our plan does not save water compared with March 2025. In total it uses 27.8 m³ *less* treated water. The GA moves water to Aulas 2 only because that combination best fills the 1410 m³ limit, not because Aulas 2 needs it. So this change is **not proven sustainable**.

- **Hydraulic infrastructure:** Aulas 2 would receive almost twice as much treated water as before. Its pipes, pumps and storage may not be sized for that. Also, 80 % reuse only works if 80 % of the building's water goes to uses like toilets or cleaning. Drinking and handwashing need potable water. Aulas 2 has never been above 42 %.
- **Energy:** the plant treats about the same total volume, but pumping water to Aulas 2 may cost more energy if it is farther or higher. Our model does not include energy, so it cannot show a benefit.
- **Service equity:** Biblioteca still gets all its water (more of it potable), so its service is not cut. However, most of the risk moves to one building. If the plant or the pipes fail, Aulas 2 users are the most affected.
- **Cedetec 65 %:** Cedetec already reached 65 % in practice, but our model only allows up to 60 %. This shows the model's limits are assumptions, not real limits. The plan lowers a building that has proven it can reuse more, and it pushes Aulas 2 to a level never observed.

**Conclusion:** before applying this plan, we should set each building's limits from real data (for example, allow 65 % in Cedetec), take pumping energy into account, and increase Aulas 2 gradually (50 → 60 → 70 %) while measuring results.